In [3]:
spark

In [8]:
print("Versión de Spark:", spark.version)
print("Master:", spark.sparkContext.master)
print("Nombre de la aplicación:", spark.sparkContext.appName)

Versión de Spark: 4.1.1
Master: local[*]
Nombre de la aplicación: test


In [7]:
df = spark.read.csv("C:/Users/DAY/Downloads/kc_house_data (1).csv", header=True, inferSchema=True)
df.show(5)

+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|        id|           date| price|bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|    lat|    long|sqft_living15|sqft_lot15|
+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|7129300520|20141013T000000|221900|       3|      1.0|       1180|    5650|   1.0|         0|   0|        3|    7|      1180|            0|    1955|           0|  98178|47.5112|-122.257|         1340|      5650|
|6414100192|20141209T000000|538000|       3|     2.25|       2570|    7242|   2.0|         0|   0|        3|    7|      2170|          400|    1951|    

In [9]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- date: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- bedrooms: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- sqft_living: integer (nullable = true)
 |-- sqft_lot: integer (nullable = true)
 |-- floors: double (nullable = true)
 |-- waterfront: integer (nullable = true)
 |-- view: integer (nullable = true)
 |-- condition: integer (nullable = true)
 |-- grade: integer (nullable = true)
 |-- sqft_above: integer (nullable = true)
 |-- sqft_basement: integer (nullable = true)
 |-- yr_built: integer (nullable = true)
 |-- yr_renovated: integer (nullable = true)
 |-- zipcode: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- sqft_living15: integer (nullable = true)
 |-- sqft_lot15: integer (nullable = true)



In [11]:
df.orderBy("zipcode").show(40, truncate=False)

+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|id        |date           |price |bedrooms|bathrooms|sqft_living|sqft_lot|floors|waterfront|view|condition|grade|sqft_above|sqft_basement|yr_built|yr_renovated|zipcode|lat    |long    |sqft_living15|sqft_lot15|
+----------+---------------+------+--------+---------+-----------+--------+------+----------+----+---------+-----+----------+-------------+--------+------------+-------+-------+--------+-------------+----------+
|5651500140|20140617T000000|272000|3       |2.0      |1380       |7476    |1.0   |0         |0   |3        |7    |1380      |0            |1989    |0           |98001  |47.3336|-122.272|1600         |7227      |
|1312200080|20140528T000000|224000|3       |1.5      |1560       |7300    |1.0   |0         |0   |4        |7    |1040      |520          |1964    |0   

In [12]:
#Zipcode con mayor número de casas
from pyspark.sql.functions import count

df_zip = df.groupBy("zipcode").agg(count("*").alias("num_casas"))
df_zip.orderBy(df_zip["num_casas"].desc()).show(10)

+-------+---------+
|zipcode|num_casas|
+-------+---------+
|  98103|      602|
|  98038|      590|
|  98115|      583|
|  98052|      574|
|  98117|      553|
|  98042|      548|
|  98034|      545|
|  98118|      508|
|  98023|      499|
|  98006|      498|
+-------+---------+
only showing top 10 rows


In [13]:
top_zip = df_zip.orderBy(df_zip["num_casas"].desc()).first()["zipcode"]
print("El zipcode con mayor número de casas es:", top_zip)

El zipcode con mayor número de casas es: 98103


In [14]:
#Promedio de precio y tamaño en m2
#Se tiene que hacer la conversión de unidades
from pyspark.sql.functions import avg, col

df.filter(df["zipcode"] == top_zip).agg(avg("price").alias("precio_promedio"),avg(col("sqft_living") * 0.092903).alias("tamano_promedio_m2")).show()

+-----------------+------------------+
|  precio_promedio|tamano_promedio_m2|
+-----------------+------------------+
|585048.7790697674|153.36711196013292|
+-----------------+------------------+



In [18]:
#Número de habitaciones y baños del precio, por zipcode
df.groupBy("zipcode", "bedrooms", "bathrooms").agg(avg("price").alias("precio_promedio")).orderBy(
    "zipcode", "bedrooms", "bathrooms").show(50, truncate=False)

+-------+--------+---------+------------------+
|zipcode|bedrooms|bathrooms|precio_promedio   |
+-------+--------+---------+------------------+
|98001  |0       |0.0      |139950.0          |
|98001  |1       |1.0      |166000.0          |
|98001  |1       |2.0      |171000.0          |
|98001  |2       |1.0      |197428.57142857142|
|98001  |2       |1.5      |350000.0          |
|98001  |2       |1.75     |246112.5          |
|98001  |2       |2.5      |214100.0          |
|98001  |2       |2.75     |239475.0          |
|98001  |3       |0.75     |363000.0          |
|98001  |3       |1.0      |205182.80952380953|
|98001  |3       |1.5      |224108.5          |
|98001  |3       |1.75     |260531.0810810811 |
|98001  |3       |2.0      |256841.42857142858|
|98001  |3       |2.25     |265999.0          |
|98001  |3       |2.5      |308581.8604651163 |
|98001  |3       |2.75     |255000.0          |
|98001  |3       |3.0      |309500.0          |
|98001  |4       |1.0      |229790.0    